# Setup

## Import

In [1]:
import json
import logging
from pathlib import Path
from torch.utils.data import DataLoader
from datasets import load_from_disk
from model_testing.model import build_model_and_transforms, get_device, CollateFn
from model_testing.utils import run_inference, compute_metrics

## Costanti

In [2]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
# Constants
IMAGENET_1K_DATASET_PATH = Path("../preprocessed_datasets/imagenet_1k_preprocessed")
RESULTS_PATH = Path("../baseline_metrics.json")
BATCH_SIZE = 64
NUM_WORKERS = 4
TOP_K = 5

# Test

## Setup modello e dataset

### Load device

In [3]:
device = get_device()
logger.info(f"Utilizing device: {device}")

INFO:__main__:Utilizing device: cuda


### Load dataset

In [4]:
if not IMAGENET_1K_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed dataset not found in {IMAGENET_1K_DATASET_PATH}. "
        "Please, execute the preprocessing script first."
    )

logger.info("Loading preprocessed dataset...")
dataset = load_from_disk(str(IMAGENET_1K_DATASET_PATH))

INFO:__main__:Loading preprocessed dataset...


### Load model

In [5]:
logger.info("Loading ResNet50...")
model, preprocess = build_model_and_transforms()
model.to(device)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=CollateFn(preprocess=preprocess),
)

INFO:__main__:Loading ResNet50...


## Inferenza

In [6]:
y_true, y_pred, topk_correct = run_inference(model, dataloader, device, k=TOP_K)

Baseline inference on imagenet-1k: 100%|██████████| 157/157 [00:19<00:00,  8.25it/s]


## Calcolo metriche

In [7]:
metrics = compute_metrics(y_true, y_pred, topk_correct)

### Stampa e salvataggio metriche

In [8]:
logger.info("Baseline metrics computed:")
for key, value in metrics.items():
    logger.info(f"  {key}: {value}")

with open(RESULTS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved in {RESULTS_PATH}")

INFO:__main__:Baseline metrics computed:
INFO:__main__:  accuracy: 0.8546
INFO:__main__:  top5_accuracy: 0.9668
INFO:__main__:  precision_macro: 0.2866908368512706
INFO:__main__:  recall_macro: 0.252094395280236
INFO:__main__:  f1_macro: 0.2665751213134408
INFO:__main__:  precision_weighted: 0.9718819369258075
INFO:__main__:  recall_weighted: 0.8546
INFO:__main__:  f1_weighted: 0.9036896612525644
INFO:__main__:  n_samples: 10000
INFO:__main__:  n_classes_present: 200
INFO:__main__:Metrics saved in ../baseline_metrics.json
